---

# Pipeline Final KNN

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Data Understanding**](#3)
4. [**Training and Validation Split**](#4)
5. [**Data Preparation**](#5)
6. [**Model Pipeline**](#6)
7. [**Modeling and Validation**](#7)
8. [**Comparison with scikit-learn**](#8)
9. [**Saving and Loading Models**](#9)
10. [**Kaggle Submission**](#10)
11. [**Error Analysis**](#11)
12. [**Cost-Sensitive Classification**](#12)
13. [**Bonus Eksperimen Biaya KNN**](#13)

---

# Introduction <a name="1"></a>

---

Notebook ini memprediksi pembatalan reservasi hotel dengan k-NN dari nol. Semua konfigurasi dibandingkan pada split validasi yang sama, sedangkan data uji hanya digunakan untuk submission.

Konfigurasi akhir memakai 21 tetangga, jarak Manhattan, bobot jarak, dan ambang 0,5. Eksperimen awal membandingkan metrik jarak, bobot kategori, fitur turunan, serta ambang tanpa mengubah split.

## Metric

Metrik utama adalah *Macro F1* pada validation dengan kedua kelas diberi bobot sama.

$$\mathrm{MacroF1}=\frac{1}{2}\sum_{c\in\{0,1\}}\mathrm{F1}_c$$

Gunakan `labels=[0, 1]`, `average='macro'`, dan `zero_division=0`.

## Dataset

Input berasal dari `data/train.csv` dan `data/test.csv`. `is_canceled` adalah target, sedangkan `id` dipertahankan untuk submission dan dikeluarkan dari fitur.

Data validasi dibuat dengan seed 42 dan mengambil sekitar 20 persen dari tiap kelas. Indeks split yang sama dipakai untuk seluruh evaluasi. Statistik prapemrosesan hanya dipelajari dari bagian latih.

Target `is_canceled` tidak digunakan sebagai fitur dan `id` hanya dipakai untuk menjaga identitas baris pada submission. Data test tidak memiliki label, sehingga tidak digunakan untuk pemilihan konfigurasi maupun evaluasi.

---

# Initialization <a name="2"></a>

---

Bagian ini menyiapkan library, konfigurasi satu model, file data, dan split bersama.

## Environment Setup

Uncomment installation only when the environment is missing these libraries.

In [1]:
# %pip install -q numpy pandas scikit-learn

Model k-NN dan prapemroses berasal dari modul proyek. NumPy, pandas, dan scikit-learn digunakan untuk perhitungan dan pembandingan.

## Import Libraries

Keep the imports needed by this model and its comparator in this cell.

In [2]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.neighbors import KNeighborsClassifier

ROOT_DIR = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "data" / "train.csv").is_file()),
    None,
)
if ROOT_DIR is None:
    raise FileNotFoundError("Jalankan notebook dari direktori proyek")
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.experiments.knn.knn import KNNClassifier
from src.experiments.knn.preprocessing import KNNPreprocessor
from src.experiments.knn.run_cost_sensitive_experiment import evaluate_cost_sensitive
from src.experiments.knn.run_experiments import make_split



## Settings

In [3]:
SEED = 42
BATCH_SIZE = 2000
FINAL_CONFIG = {
    "n_neighbors": 21,
    "weights": "distance",
    "metric": "manhattan",
    "cat_weight": 1.0,
    "threshold": 0.5,
}
DATA_DIR = ROOT_DIR / "data"
OUTPUT_DIR = ROOT_DIR / "output"
KNN_OUTPUT_DIR = OUTPUT_DIR / "knn"
MODEL_DIR = OUTPUT_DIR / "final" / "model" / "knn"

print(f"NumPy {np.__version__} dan pandas {pd.__version__}")

NumPy 1.26.4 dan pandas 2.2.2


## Load Dataset

In [4]:
train_df = pd.read_csv(DATA_DIR / "train.csv", dtype={"id": str})
test_df = pd.read_csv(DATA_DIR / "test.csv", dtype={"id": str})

required_train = {"id", "is_canceled"}
if not required_train.issubset(train_df.columns) or "id" not in test_df.columns:
    raise ValueError("Kolom id atau target tidak ditemukan")
if not train_df["id"].is_unique or not test_df["id"].is_unique:
    raise ValueError("Kolom id harus unik pada train dan test")

feature_columns = [
    column for column in train_df.columns if column not in {"id", "is_canceled"}
]
if feature_columns != [column for column in test_df.columns if column != "id"]:
    raise ValueError("Fitur train dan test tidak cocok atau berbeda urutan")

y = train_df["is_canceled"].to_numpy(dtype=np.int64)
if not np.isin(y, [0, 1]).all():
    raise ValueError("Target hanya boleh berisi label 0 dan 1")
train_idx, valid_idx = make_split(y)
X_raw = train_df[feature_columns].copy()
X_test_raw = test_df[feature_columns].copy()
y_train, y_valid = y[train_idx], y[valid_idx]

print(f"Train berisi {len(train_idx)} baris")
print(f"Validasi berisi {len(valid_idx)} baris")
display(pd.DataFrame({
    "train": pd.Series(y_train).value_counts().reindex([0, 1], fill_value=0),
    "validation": pd.Series(y_valid).value_counts().reindex([0, 1], fill_value=0),
}))
train_set = train_df.iloc[train_idx].copy()
val_set = train_df.iloc[valid_idx].copy()


Train berisi 21898 baris
Validasi berisi 5475 baris


,train,validation
0,14723,3681
1,7175,1794


Split dengan seed 42 menggunakan sekitar 20% dari tiap kelas sebagai validation. Jangan ubah indeks ini saat membandingkan konfigurasi.

---

# Data Understanding <a name="3"></a>

---

Gunakan bagian ini untuk memahami data yang tersedia bagi model ini. Pemeriksaan yang memakai target dilakukan hanya pada `train_set`.

## Describe Data

In [5]:
train_features = train_set[feature_columns]
feature_profile = pd.DataFrame({
    "dtype": X_raw.dtypes.astype(str),
    "missing_train": X_raw.iloc[train_idx].isna().sum(),
    "missing_validation": X_raw.iloc[valid_idx].isna().sum(),
    "unique_train": X_raw.iloc[train_idx].nunique(dropna=True),
})
display(feature_profile)
print(f"Jumlah fitur mentah {len(feature_columns)}")
print(f"Jumlah baris train {len(train_df)} dan test {len(test_df)}")

,dtype,missing_train,missing_validation,unique_train
num_adults,float64,652,157,5
num_children,float64,2453,582,6
weekend_nights,int64,0,0,8
weekday_nights,int64,0,0,18
meal_package,object,1413,347,4
needs_parking,float64,865,217,2
room_category,object,811,191,7
days_before_arrival,float64,909,214,403
arrival_year,int64,0,0,2
arrival_month,int64,0,0,12


Jumlah fitur mentah 17
Jumlah baris train 27373 dan test 6844


Ringkasan ini tidak mengubah data. Gunakan hanya untuk merumuskan eksperimen pipeline model ini, bukan untuk melihat label validation.

## Explore Data

In [6]:
numeric_features = train_features.select_dtypes(include='number').columns
categorical_features = train_features.select_dtypes(exclude='number').columns

display(train_set.groupby('is_canceled')[numeric_features].median().T.rename(columns={0: 'not_canceled', 1: 'canceled'}))
for column in categorical_features:
    category_summary = train_set.assign(category=train_set[column].fillna('__MISSING__')).groupby('category', dropna=False)['is_canceled'].agg(count='size', cancellation_rate='mean')
    display(category_summary.sort_values('count', ascending=False))

is_canceled,not_canceled,canceled
num_adults,2.0,2.0
num_children,0.0,0.0
weekend_nights,1.0,1.0
weekday_nights,2.0,2.0
needs_parking,0.0,0.0
days_before_arrival,39.0,122.0
arrival_year,2024.0,2024.0
arrival_month,8.0,8.0
arrival_day,15.0,16.0
is_returning_guest,0.0,0.0


,count,cancellation_rate
category,,
Breakfast,15716,0.310957
Room Only,2981,0.338812
Half Board,1784,0.459081
__MISSING__,1413,0.324133
Full Board,4,0.250000


,count,cancellation_rate
category,,
Type B,16325,0.320674
Type G,3507,0.341888
__MISSING__,811,0.361282
Type A,572,0.430070
Type D,429,0.328671
Type C,152,0.236842
Type F,96,0.239583
Type E,6,0.333333


,count,cancellation_rate
category,,
Online Travel Agent,14036,0.363138
Offline Agent,6357,0.301715
Corporate,1189,0.116905
Complimentary,241,0.000000
Airline,75,0.280000


Ringkasan berdasarkan target hanya memakai `train_set`. Perbedaan antarkelas dipakai untuk menyusun hipotesis, bukan memilih parameter dari data uji.

## Verify Data Quality

In [7]:
test_features = test_df.drop(columns='id')
quality_checks = {
    'unique IDs in train': train_df['id'].is_unique,
    'unique IDs in test': test_df['id'].is_unique,
    'train and test feature columns match': set(feature_columns) == set(test_features.columns),
    'binary training target': set(train_set['is_canceled'].unique()).issubset({0, 1}),
}
display(pd.Series(quality_checks, name='passed').to_frame())
if not all(quality_checks.values()):
    raise ValueError('Review data quality checks before continuing.')

,passed
unique IDs in train,True
unique IDs in test,True
train and test feature columns match,True
binary training target,True


Keunikan ID, label biner, dan kesesuaian kolom diperiksa sebelum pelatihan. Pemeriksaan gagal menghentikan notebook.

---

# Training and Validation Split <a name="4"></a>

---

Split sudah dibuat di Initialization sebelum eksplorasi target-aware agar `train_set` dapat digunakan pada bagian Data Understanding.

In [8]:
split_class_counts = pd.DataFrame({
    'train': train_set['is_canceled'].value_counts().reindex([0, 1], fill_value=0),
    'validation': val_set['is_canceled'].value_counts().reindex([0, 1], fill_value=0),
})
display(split_class_counts)

cross_split_duplicates = train_set[feature_columns].drop_duplicates().merge(
    val_set[feature_columns].drop_duplicates(), on=feature_columns, how='inner'
).shape[0]
print(f'Identical feature rows across train and validation: {cross_split_duplicates}')

,train,validation
is_canceled,,
0,14723,3681
1,7175,1794


Identical feature rows across train and validation: 25


Laporkan jumlah contoh per kelas dan duplikasi lintas split. Jangan mengubah split setelah melihat skor validation.

---

# Data Preparation <a name="5"></a>

---

Fitur numerik selain dua hitungan riwayat dibakukan dengan rata-rata dan simpangan baku data latih. `prior_cancellations` dan `prior_completed_bookings` diubah menjadi indikator di atas median latih. Fitur kategorikal diubah menjadi one-hot, termasuk kolom untuk nilai hilang dan kategori yang belum pernah terlihat.

Prapemroses hanya di-fit pada bagian latih. Transformasi validasi memakai statistik dan kategori yang sama. Bobot kategori mengatur kontribusi blok kategorikal terhadap jarak.

## Select Data

In [9]:
X_train_raw = train_set[feature_columns].copy()
X_val_raw = val_set[feature_columns].copy()
X_test_raw = test_df[feature_columns].copy()
y_train = train_set['is_canceled'].to_numpy(dtype=int)
y_val = val_set['is_canceled'].to_numpy(dtype=int)

if X_train_raw.columns.tolist() != X_val_raw.columns.tolist() or X_train_raw.columns.tolist() != X_test_raw.columns.tolist():
    raise ValueError('Train, validation, and test feature columns must match in the same order.')
if {'id', 'is_canceled'} & set(feature_columns):
    raise ValueError('ID and target must not be used as model features.')
print(f'Features: {len(feature_columns)} | train: {len(X_train_raw)} | validation: {len(X_val_raw)} | test: {len(X_test_raw)}')

Features: 17 | train: 21898 | validation: 5475 | test: 6844


`id` disimpan untuk submission dan tidak menjadi fitur. Urutan baris tetap sama selama prapemrosesan dan prediksi.

## Clean Data

### Missing Values

In [10]:
missing_summary = pd.DataFrame({
    'train': X_train_raw.isna().sum(),
    'validation': X_val_raw.isna().sum(),
    'test': X_test_raw.isna().sum(),
})
display(missing_summary.loc[missing_summary.any(axis=1)].sort_values('train', ascending=False))

,train,validation,test
num_children,2453,582,747
special_requests,2029,471,593
nightly_rate_idr,1640,398,540
meal_package,1413,347,442
days_before_arrival,909,214,290
needs_parking,865,217,247
room_category,811,191,242
num_adults,652,157,180


Nilai kosong numerik dan kategorikal ditangani oleh `KNNPreprocessor` yang di-fit pada data latih saja.

### Duplicates

In [11]:
duplicate_mask = X_train_raw.duplicated(subset=feature_columns, keep=False)
print(f'Training rows with duplicate features: {int(duplicate_mask.sum())}')

Training rows with duplicate features: 79


Duplikasi dilaporkan tetapi tidak dihapus otomatis, terutama bila label pada fitur identik berbeda.

### Outliers

In [12]:
numeric_columns = X_train_raw.select_dtypes(include='number').columns
quartiles = X_train_raw[numeric_columns].quantile([0.25, 0.75])
iqr = quartiles.loc[0.75] - quartiles.loc[0.25]
lower, upper = quartiles.loc[0.25] - 1.5 * iqr, quartiles.loc[0.75] + 1.5 * iqr
outlier_counts = ((X_train_raw[numeric_columns] < lower) | (X_train_raw[numeric_columns] > upper)).sum()
display(outlier_counts.rename('iqr_outliers').to_frame().sort_values('iqr_outliers', ascending=False))

,iqr_outliers
num_adults,5850
arrival_year,3925
num_children,1626
nightly_rate_idr,924
days_before_arrival,739
needs_parking,637
is_returning_guest,546
prior_completed_bookings,476
special_requests,436
weekday_nights,200


Batas IQR hanya untuk diagnosis. Tentukan penanganannya berdasarkan algoritma dan validasi, bukan dengan clipping otomatis.

### Data Errors

Catat nilai di luar domain, tipe yang tidak sesuai, atau ketidakkonsistenan yang relevan untuk model ini sebelum mengubah data.

## Construct Data

### Attribute Construction

Tidak ada fitur turunan yang diadopsi. `total_guests` dan `total_stay_value` menurunkan skor validasi dibandingkan fitur dasar.

### Feature Scaling

Fitur numerik selain dua hitungan riwayat dibakukan memakai rata-rata dan simpangan baku data latih. Hitungan riwayat menjadi indikator berdasarkan median latih.

### Encoding

Kategori menjadi kolom one-hot, termasuk nilai hilang dan kategori yang belum terlihat. Bobot blok kategorikal tetap 1,0 sesuai konfigurasi akhir.

### Dimensionality Reduction

Tidak ada reduksi dimensi. Fitur turunan yang diuji tidak diadopsi karena tidak melewati batas peningkatan skor.

## Class Balance

Tidak dilakukan resampling atau pembobotan kelas. Macro F1 memberi bobot evaluasi yang sama untuk kelas 0 dan 1.

## Format Data

Prapemroses menghasilkan matriks numerik dengan urutan baris tetap. Model dari nol dan pembanding scikit-learn menerima matriks yang sama.

---

# Model Pipeline <a name="6"></a>

---

Prapemroses dipelajari hanya dari bagian latih. Validasi dan data uji ditransformasi dengan state yang sama.

In [13]:
preprocessor = KNNPreprocessor(
    cat_weight=FINAL_CONFIG["cat_weight"]
).fit(X_raw.iloc[train_idx])
X_train_model = preprocessor.transform(X_raw.iloc[train_idx])
X_valid_model = preprocessor.transform(X_raw.iloc[valid_idx])



Rata-rata, simpangan baku, median, dan kategori dipelajari hanya dari data latih. Validasi dan data uji memakai state tersebut.

---

# Modeling and Validation <a name="7"></a>

---

Grid nilai k dan pencarian ambang tidak melampaui batas adopsi 0,005, sehingga nilai awal k 21 dan ambang 0,5 dipertahankan. Penggantian metrik Euclidean menjadi Manhattan menaikkan Macro F1 dari 0,831783 menjadi 0,841689. Bobot kategori 0,5 menghasilkan 0,843450, tetapi peningkatannya belum melewati batas adopsi.

In [14]:
selected_config = FINAL_CONFIG


Hasil ini membandingkan satu perubahan pada satu tahap dari proses seleksi yang dilakukan. Skor validasi dipakai untuk memilih konfigurasi dan tidak menjadi estimasi evaluasi independen.

In [15]:
model = KNNClassifier(
    n_neighbors=FINAL_CONFIG["n_neighbors"],
    weights=FINAL_CONFIG["weights"],
    metric=FINAL_CONFIG["metric"],
    threshold=FINAL_CONFIG["threshold"],
).fit(X_train_model, y_train)
valid_scores = model.decision_scores(X_valid_model, batch_size=BATCH_SIZE)
valid_predictions = (valid_scores > FINAL_CONFIG["threshold"]).astype(np.int64)
if valid_predictions.shape != y_valid.shape:
    raise ValueError("Prediksi validasi tidak memiliki satu label per baris")

validation_macro_f1 = f1_score(
    y_valid, valid_predictions, labels=[0, 1], average="macro", zero_division=0
)
validation_matrix = confusion_matrix(y_valid, valid_predictions, labels=[0, 1])
precision, recall, class_f1, support = precision_recall_fscore_support(
    y_valid, valid_predictions, labels=[0, 1], zero_division=0
)
validation_metrics = pd.DataFrame({
    "precision": precision,
    "recall": recall,
    "f1": class_f1,
    "support": support,
}, index=pd.Index([0, 1], name="kelas"))
print(f"Macro F1 validasi {validation_macro_f1:.6f}")
display(validation_metrics)
display(pd.DataFrame(
    validation_matrix,
    index=pd.Index([0, 1], name="aktual"),
    columns=pd.Index([0, 1], name="prediksi"),
))

Macro F1 validasi 0.841689


,precision,recall,f1,support
kelas,,,,
0,0.881695,0.921217,0.901023,3681
1,0.821977,0.746377,0.782355,1794


prediksi,0,1
aktual,,
0,3391,290
1,455,1339


Macro F1 0.841689 menjadi acuan konfigurasi final. Pada matriks validasi tercatat 290 false positive dan 455 false negative. Precision kelas batal sebesar 0.8220 dan recall kelas batal sebesar 0.7464. Skor merangkum keseimbangan kedua kelas, sedangkan matriks memperlihatkan jenis kesalahan yang masih terjadi.

---

# Comparison with scikit-learn <a name="8"></a>

---

Pembanding memakai fitur hasil prapemrosesan yang sama, indeks split yang sama, dan parameter tetangga yang sama. Implementasi dari nol tetap menjadi model yang dievaluasi untuk pipeline ini. Perbedaan kecil dapat timbul dari aturan pemilihan tetangga ketika jarak sama.

In [16]:
sklearn_model = KNeighborsClassifier(
    n_neighbors=FINAL_CONFIG["n_neighbors"],
    weights=FINAL_CONFIG["weights"],
    metric=FINAL_CONFIG["metric"],
    n_jobs=-1,
).fit(X_train_model, y_train)
sklearn_scores = sklearn_model.predict_proba(X_valid_model)[:, 1]
sklearn_predictions = (sklearn_scores > FINAL_CONFIG["threshold"]).astype(np.int64)
sklearn_macro_f1 = f1_score(
    y_valid, sklearn_predictions, labels=[0, 1], average="macro", zero_division=0
)
display(pd.DataFrame({
    "model": ["KNN dari nol", "scikit-learn"],
    "macro_f1": [validation_macro_f1, sklearn_macro_f1],
    "beda_prediksi": [
        0,
        int(np.count_nonzero(valid_predictions != sklearn_predictions)),
    ],
}))

,model,macro_f1,beda_prediksi
0,KNN dari nol,0.841689,0
1,scikit-learn,0.841689,0


Kedua implementasi memakai fitur dan indeks validasi yang sama. Selisih prediksi dapat muncul jika ada jarak tetangga yang seri.

---

# Saving and Loading Models <a name="9"></a>

---

Setelah konfigurasi ditetapkan, prapemroses dan model di-fit ulang menggunakan seluruh data train. Prediksi test mengikuti urutan `id` pada berkas sumber. Berkas model menyimpan data latih terproses, label, konfigurasi, dan state prapemrosesan agar prediksi dapat direkonstruksi.

In [17]:
final_preprocessor = KNNPreprocessor(
    cat_weight=FINAL_CONFIG["cat_weight"]
).fit(X_raw)
X_full = final_preprocessor.transform(X_raw)
X_test = final_preprocessor.transform(X_test_raw)
final_model = KNNClassifier(
    n_neighbors=FINAL_CONFIG["n_neighbors"],
    weights=FINAL_CONFIG["weights"],
    metric=FINAL_CONFIG["metric"],
    threshold=FINAL_CONFIG["threshold"],
).fit(X_full, y)
test_predictions = final_model.predict(X_test, batch_size=BATCH_SIZE)
if len(test_predictions) != len(test_df) or not np.isin(test_predictions, [0, 1]).all():
    raise ValueError("Prediksi test harus berisi satu label biner per baris")

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    MODEL_DIR / "knn_model.npz",
    X_train=final_model.X_train,
    y_train=final_model.y_train,
)
(MODEL_DIR / "preprocessing.json").write_text(
    json.dumps(final_preprocessor.to_dict(), ensure_ascii=False, indent=2),
    encoding="utf-8",
)
model_config = {
    "algorithm": "KNN from scratch",
    "k": FINAL_CONFIG["n_neighbors"],
    "weights": FINAL_CONFIG["weights"],
    "metric": FINAL_CONFIG["metric"],
    "cat_weight": FINAL_CONFIG["cat_weight"],
    "threshold": FINAL_CONFIG["threshold"],
    "seed": SEED,
    "macro_f1_validasi": round(float(validation_macro_f1), 6),
    "feature_engineering": "tidak ada fitur turunan yang diadopsi",
}
(MODEL_DIR / "konfigurasi.json").write_text(
    json.dumps(model_config, ensure_ascii=False, indent=2),
    encoding="utf-8",
)



252

In [18]:
saved_arrays = np.load(MODEL_DIR / "knn_model.npz")
saved_preprocessor = KNNPreprocessor.from_dict(json.loads(
    (MODEL_DIR / "preprocessing.json").read_text(encoding="utf-8")
))
saved_config = json.loads(
    (MODEL_DIR / "konfigurasi.json").read_text(encoding="utf-8")
)
reloaded_model = KNNClassifier(
    n_neighbors=saved_config["k"],
    weights=saved_config["weights"],
    metric=saved_config["metric"],
    threshold=saved_config["threshold"],
).fit(saved_arrays["X_train"], saved_arrays["y_train"])
reloaded_predictions = reloaded_model.predict(
    saved_preprocessor.transform(X_test_raw), batch_size=BATCH_SIZE
)
np.testing.assert_array_equal(reloaded_predictions, test_predictions)
print("Pemeriksaan muat ulang berhasil dan prediksi tetap identik")

Pemeriksaan muat ulang berhasil dan prediksi tetap identik


Model dan prapemroses dimuat ulang dari berkas. Prediksinya diuji identik dengan prediksi sebelum penyimpanan.

---

# Kaggle Submission <a name="10"></a>

---

Prediksi test dibuat dari model dan preprocessing yang baru dimuat, dengan urutan `id` persis seperti `test.csv`.

In [19]:
test_predictions = reloaded_predictions
submission = pd.DataFrame({
    "id": test_df["id"].to_numpy(),
    "is_canceled": test_predictions.astype(np.int64),
})
if submission.columns.tolist() != ["id", "is_canceled"]:
    raise ValueError("Kolom submission tidak sesuai kontrak")
if not submission["id"].equals(test_df["id"].reset_index(drop=True)):
    raise ValueError("Urutan id submission berubah")
KNN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
submission_path = KNN_OUTPUT_DIR / "submission_knn.csv"
submission.to_csv(submission_path, index=False)
print(f"Model tersimpan di {MODEL_DIR}")
print(f"Submission tersimpan di {submission_path} dengan {len(submission)} baris")
display(submission["is_canceled"].value_counts().sort_index().rename("jumlah"))

Model tersimpan di c:\Users\aufar\Institut Teknologi Bandung\IF3170_G03_FiveAgents\output\final\model\knn
Submission tersimpan di c:\Users\aufar\Institut Teknologi Bandung\IF3170_G03_FiveAgents\output\knn\submission_knn.csv dengan 6844 baris


is_canceled
0    4837
1    2007
Name: jumlah, dtype: int64

Submission berisi hanya `id,is_canceled`, satu prediksi integer untuk setiap baris test.

---

# Error Analysis <a name="11"></a>

---

Kesalahan berikut berasal dari split validasi dan dapat dipakai untuk merumuskan eksperimen lanjutan. Label validasi tidak dimasukkan kembali ke model setelah konfigurasi dipilih. Data test tidak ditampilkan karena labelnya tidak tersedia.

In [20]:
error_analysis = X_raw.iloc[valid_idx].reset_index(drop=True).copy()
error_analysis["actual"] = y_valid
error_analysis["predicted"] = valid_predictions
error_analysis["error_type"] = np.select(
    [
        error_analysis["actual"].eq(0) & error_analysis["predicted"].eq(1),
        error_analysis["actual"].eq(1) & error_analysis["predicted"].eq(0),
    ],
    ["false_positive", "false_negative"],
    default="correct",
)
display(error_analysis["error_type"].value_counts())
display(error_analysis.loc[error_analysis["error_type"] != "correct"].head(20))

error_type
correct           4730
false_negative     455
false_positive     290
Name: count, dtype: int64

,num_adults,num_children,weekend_nights,weekday_nights,meal_package,needs_parking,room_category,days_before_arrival,arrival_year,arrival_month,arrival_day,booking_channel,is_returning_guest,prior_cancellations,prior_completed_bookings,nightly_rate_idr,special_requests,actual,predicted,error_type
26,2.0,0.0,0,1,Breakfast,0.0,Type G,88.0,2024,9,15,Online Travel Agent,0,0,0,2395000.0,1.0,1,0,false_negative
27,2.0,0.0,0,3,Breakfast,0.0,Type G,125.0,2024,9,28,Online Travel Agent,0,0,0,2301000.0,2.0,1,0,false_negative
37,2.0,0.0,0,3,NaN,NaN,Type G,6.0,2024,3,22,Online Travel Agent,0,0,0,2492000.0,0.0,0,1,false_positive
39,2.0,0.0,0,3,Room Only,0.0,Type B,120.0,2023,7,30,Online Travel Agent,0,0,0,927000.0,0.0,1,0,false_negative
40,2.0,0.0,2,2,Half Board,0.0,Type B,164.0,2024,10,28,Online Travel Agent,0,0,0,2100000.0,2.0,1,0,false_negative
55,3.0,0.0,2,2,Breakfast,0.0,Type G,81.0,2024,10,15,Online Travel Agent,0,0,0,2524000.0,1.0,0,1,false_positive
62,1.0,0.0,0,1,Breakfast,0.0,Type B,14.0,2024,4,7,Online Travel Agent,0,0,0,1686000.0,NaN,0,1,false_positive
68,2.0,2.0,0,4,Breakfast,0.0,Type A,167.0,2024,7,4,Online Travel Agent,0,0,0,2799000.0,2.0,1,0,false_negative
78,2.0,0.0,1,5,Breakfast,0.0,Type B,39.0,2024,12,5,Online Travel Agent,0,0,0,1391000.0,0.0,1,0,false_negative
87,2.0,1.0,0,4,Room Only,0.0,Type B,5.0,2024,3,1,Online Travel Agent,0,0,0,1255000.0,0.0,0,1,false_positive


Contoh kesalahan pada split validasi membantu merumuskan eksperimen baru. Data uji tidak memiliki label untuk analisis kesalahan.

---

# Cost-Sensitive Classification <a name="12"></a>

---

Jika FP berbiaya dua kali FN, ambang keputusan teoretis menjadi 2/3 hanya untuk probabilitas yang terkalibrasi. Skor voting k-NN belum tentu terkalibrasi. Ambang lebih tinggi cenderung mengurangi FP dengan risiko menambah FN.

Eksperimen utama pada tabel memakai rasio tiga banding satu dan memilih ambang melalui tiga lipatan data latih, lalu mengukurnya pada holdout. Rasio dua banding satu dianalisis terpisah pada bagian berikutnya. Jalankan `python -m src.experiments.knn.run_cost_sensitive_experiment` untuk menghasilkan tabel perbandingan.

In [21]:
cost_path = ROOT_DIR / 'output/knn/cost_sensitive_comparison.csv'
if cost_path.exists():
    display(pd.read_csv(cost_path))
else:
    print('Jalankan eksperimen biaya KNN terlebih dahulu.')

,strategy,threshold,fp,fn,cost,macro_f1
0,baseline,0.500000,290,455,1325.0,0.841689
1,selected,0.693585,86,797,1055.0,0.791853
2,oof_selected,0.693585,386,3400,4558.0,0.774692


---

# Bonus Eksperimen Biaya KNN <a name="13"></a>

---

Analisis tambahan memberi biaya dua untuk false positive dan satu untuk false negative. Ambang teoretis 2/3 berlaku jika skor dapat ditafsirkan sebagai probabilitas terkalibrasi. Skor voting KNN belum tentu memenuhi asumsi tersebut, sehingga ambang dipilih secara empiris pada validasi dan hasilnya dibaca sebagai analisis sensitivitas.

In [22]:
cost_results, cost_comparison = evaluate_cost_sensitive(
    y_valid, valid_scores, KNN_OUTPUT_DIR
)
display(cost_comparison[[
    "strategy", "threshold", "fp", "fn", "cost", "cost_per_booking", "macro_f1"
]])

,strategy,threshold,fp,fn,cost,cost_per_booking,macro_f1
0,baseline,0.500000,290.0,455.0,1035.0,0.189041,0.841689
1,theoretical_2_over_3,0.666667,108.0,746.0,962.0,0.175708,0.801879
2,validation_min_cost,0.600000,158.0,623.0,939.0,0.171507,0.825069


Ambang 0.60 menurunkan biaya validasi dari 1035 menjadi 939, atau 9.28 persen. False positive turun dari 290 menjadi 158, sedangkan false negative naik dari 455 menjadi 623. Macro F1 turun dari 0.841689 menjadi 0.825069. Karena ambang dipilih dan dinilai pada validasi yang sama, hasil ini dapat optimistis dan tidak menggantikan konfigurasi utama.